In [ ]:
# Project Root Check
from pathlib import Path
import os

# Step 1: Start from current working directory
CWD = Path.cwd()

# Step 2: If we are inside notebooks/, move one level up
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

# Step 3: Change directory to project root
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)

# Step 4: Sanity checks
assert (PROJECT_ROOT / "data").exists(), "data/ folder not found"
assert (PROJECT_ROOT / "notebooks").exists(), "notebooks/ folder not found"

In [2]:
from pathlib import Path

print((Path("data")).exists())

True


In [ ]:
from pathlib import Path
Project_root= Path().resolve()
print("Project root:",Project_root)

Load Raw Data

In [ ]:
import glob
import os
import pandas as pd

print("CWD:", os.getcwd())

files = glob.glob(
    r"data/PRSA_Data_20130301-20170228/PRSA_Data_*.csv"
)

print("Files found:", len(files))
print(files[:3])

if not files:
    raise FileNotFoundError("PRSA CSV files not found")

dfs = []
for f in files:
    station_id = os.path.basename(f).split("_")[2]
    temp = pd.read_csv(f)
    temp["station_id"] = station_id
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
print("Final shape:", df.shape)

In [5]:
# Standardize Date Time 

df.columns=(df.columns.str.strip().str.lower().str.replace(' ',"_"))

df['datetime']=pd.to_datetime(df[["year","month","day","hour"]])

df= df.sort_values(["station_id","datetime"])

df=df.set_index("datetime")

In [6]:
# Alert Target
PM25_THRESHOLD= 150
df["alert"]= (df["pm2.5"]>= PM25_THRESHOLD).astype(int)

In [7]:
# Lag Features

def add_lag_features(df, lags=(1, 3, 6)):
    df = df.sort_values(["station_id", "datetime"])
    for lag in lags:
        df[f"pm2.5_lag_{lag}"] = (
            df.groupby("station_id")["pm2.5"].shift(lag)
        )
    return df

df = add_lag_features(df)
df_hourly = df.dropna()


In [8]:
print(type(df_hourly.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [9]:
# Save CSVs

os.makedirs("data/processed", exist_ok=True)
df_hourly.to_csv("data/processed/hourly_features.csv")

numeric_cols = df_hourly.select_dtypes(include="number").columns
df_daily = (
    df_hourly
    .groupby("station_id")[numeric_cols]
    .resample("1D")
    .mean()
    .reset_index()
)

df_daily.to_csv("data/processed/daily_features.csv", index=False)
